In [1]:
!pip install -q pyclipper shapely editdistance


In [2]:
# ============================================================
# Imports, paths, reproducibility (SEED=42)
# ============================================================
import os, glob, re, json, random, unicodedata
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.models import resnet18, ResNet18_Weights
from tqdm import tqdm

import pyclipper
from shapely.geometry import Polygon
import editdistance

BASE      = "/kaggle/input/datasets/hungkhoi/vietnamese-scene-text-spotting-dataset-vintext/vintext/vintext"
TRAIN_DIR = os.path.join(BASE, "train_images")
VAL_DIR   = os.path.join(BASE, "val_image")
TEST_DIR  = os.path.join(BASE, "test_image")
LABEL_DIR = os.path.join(BASE, "labels")
OUT_DIR   = "/kaggle/working/dbnet_crnn_e2e_vintext"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  SEED: {SEED}")


Device: cuda  |  SEED: 42


In [ ]:
# ============================================================
# Vietnamese alphabet (NFD, 106 chars with CTC blank)

SPECIAL_BLANK = "<BLANK>"

VN_BASE_CHARS = (
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    " !\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"
    "\u0301"   # acute
    "\u0300"   # grave
    "\u0309"   # hook above
    "\u0323"   # dot below
    "\u0303"   # tilde
    "\u0302"   # circumflex
    "\u0306"   # breve
    "\u031b"   # horn
    "\u0111"   # d with stroke
    "\u0110"   # D with stroke
)

_seen, _char_set = set(), []
for c in VN_BASE_CHARS:
    if c not in _seen:
        _char_set.append(c); _seen.add(c)

CHARS       = [SPECIAL_BLANK] + _char_set
CHAR2IDX    = {c: i for i, c in enumerate(CHARS)}
IDX2CHAR    = {i: c for c, i in CHAR2IDX.items()}
NUM_CLASSES = len(CHARS)
print(f"Alphabet size (incl. CTC blank): {NUM_CLASSES}")


def text_to_indices(text: str) -> List[int]:
    text = unicodedata.normalize("NFD", text.strip())
    return [CHAR2IDX[c] for c in text if c in CHAR2IDX]


def indices_to_text(indices) -> str:
    return "".join(IDX2CHAR.get(i, "") for i in indices if i != 0)


def normalize_for_match(s: str) -> str:
    return unicodedata.normalize("NFD", s).strip()


Alphabet size (incl. CTC blank): 106


In [4]:
# ============================================================
# VinText annotation parser
# ============================================================
def parse_annotation(gt_path: str) -> List[Tuple[np.ndarray, str]]:
    records = []
    if not os.path.exists(gt_path):
        return records
    with open(gt_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",", 8)
            if len(parts) < 9:
                continue
            try:
                coords = list(map(float, parts[:8]))
            except ValueError:
                continue
            transcript = parts[8].strip()
            pts = np.array(coords, dtype=np.float32).reshape(4, 2)
            records.append((pts, transcript))
    return records


def _img_idx_from_name(name: str):
    m = re.search(r"(\d+)", os.path.basename(name))
    return int(m.group(1)) if m else None


def gather_image_records(img_dir: str, label_dir: str):
    for ip in sorted(glob.glob(os.path.join(img_dir, "*.jpg")) +
                     glob.glob(os.path.join(img_dir, "*.png"))):
        idx = _img_idx_from_name(ip)
        gt = os.path.join(label_dir, f"gt_{idx}.txt")
        records = parse_annotation(gt)
        if records:
            yield ip, records


In [5]:
# ============================================================
# 4. DBNet ground-truth generation
#    Implements the prob/threshold/mask map construction from
#    Liao et al., "Real-time Scene Text Detection with
#    Differentiable Binarization" (AAAI'20).
# ============================================================
def shrink_polygon(poly: np.ndarray, shrink_ratio: float = 0.4) -> Optional[np.ndarray]:
    """Vatti-clip shrink, used to build the prob-map GT (text core)."""
    try:
        p = Polygon(poly)
    except Exception:
        return None
    if (not p.is_valid) or p.area <= 1.0:
        return None
    distance = p.area * (1.0 - shrink_ratio ** 2) / max(p.length, 1e-6)
    pco = pyclipper.PyclipperOffset()
    try:
        pco.AddPath(np.round(poly).astype(int).tolist(),
                    pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
        shrunk = pco.Execute(-distance)
    except Exception:
        return None
    if not shrunk:
        return None
    return np.asarray(shrunk[0], dtype=np.int32)


def make_db_targets(polygons: List[np.ndarray],
                    transcripts: List[str],
                    h: int, w: int,
                    shrink_ratio: float = 0.4,
                    thresh_min: float = 0.3,
                    thresh_max: float = 0.7):
    """Build DBNet ground-truth maps for one image.

    Returns
    -------
    gt_prob       : (H, W) float32  -- 1 inside shrunk polygons, 0 elsewhere
    gt_thresh     : (H, W) float32  -- distance-decayed threshold map in [thresh_min, thresh_max]
    prob_mask     : (H, W) float32  -- 0 = ignore (e.g. ### regions), 1 = trainable
    thresh_mask   : (H, W) float32  -- 0/1 mask for the L1 threshold loss (only the boundary band)
    """
    gt_prob     = np.zeros((h, w), dtype=np.float32)
    gt_thresh   = np.zeros((h, w), dtype=np.float32)
    prob_mask   = np.ones ((h, w), dtype=np.float32)
    thresh_mask = np.zeros((h, w), dtype=np.float32)

    for poly, txt in zip(polygons, transcripts):
        if poly is None:
            continue
        ignore = txt in ("###", "***", "")
        if ignore:
            cv2.fillPoly(prob_mask, [poly.astype(np.int32)], 0.0)
            continue

        # ---- prob map: shrunk polygon core ----
        shrunk = shrink_polygon(poly, shrink_ratio)
        if shrunk is None:
            continue
        cv2.fillPoly(gt_prob, [shrunk.astype(np.int32)], 1.0)

        # ---- threshold map: distance band between shrunk and dilated polygon ----
        try:
            p = Polygon(poly)
            distance = p.area * (1.0 - shrink_ratio ** 2) / max(p.length, 1e-6)
        except Exception:
            continue
        pco = pyclipper.PyclipperOffset()
        try:
            pco.AddPath(np.round(poly).astype(int).tolist(),
                        pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
            dilated = pco.Execute(distance)
        except Exception:
            continue
        if not dilated:
            continue
        dilated_arr = np.asarray(dilated[0], dtype=np.int32)
        x_min, y_min = dilated_arr.min(axis=0)
        x_max, y_max = dilated_arr.max(axis=0)
        x_min = max(int(x_min), 0); y_min = max(int(y_min), 0)
        x_max = min(int(x_max), w - 1); y_max = min(int(y_max), h - 1)
        if x_max <= x_min or y_max <= y_min:
            continue
        xs = np.arange(x_min, x_max + 1)
        ys = np.arange(y_min, y_max + 1)
        xv, yv = np.meshgrid(xs, ys)

        # Distance from each pixel in the dilated bbox to the nearest polygon edge.
        n = poly.shape[0]
        dist_map = np.full(xv.shape, np.inf, dtype=np.float32)
        for k in range(n):
            x1, y1 = poly[k]
            x2, y2 = poly[(k + 1) % n]
            dx, dy = x2 - x1, y2 - y1
            seg_len_sq = dx * dx + dy * dy + 1e-12
            t = ((xv - x1) * dx + (yv - y1) * dy) / seg_len_sq
            t = np.clip(t, 0.0, 1.0)
            proj_x = x1 + t * dx
            proj_y = y1 + t * dy
            d = np.sqrt((xv - proj_x) ** 2 + (yv - proj_y) ** 2)
            dist_map = np.minimum(dist_map, d)
        dist_map = 1.0 - np.clip(dist_map / max(distance, 1e-6), 0.0, 1.0)

        # Restrict the threshold-loss band to the dilated polygon only.
        band_mask = np.zeros_like(dist_map, dtype=np.float32)
        cv2.fillPoly(band_mask, [dilated_arr - np.array([[x_min, y_min]])], 1.0)
        dist_map = dist_map * band_mask

        np.maximum(gt_thresh[y_min:y_max + 1, x_min:x_max + 1],
                   dist_map,
                   out=gt_thresh[y_min:y_max + 1, x_min:x_max + 1])
        thresh_mask[y_min:y_max + 1, x_min:x_max + 1] = np.maximum(
            thresh_mask[y_min:y_max + 1, x_min:x_max + 1], band_mask
        )

    gt_thresh = gt_thresh * (thresh_max - thresh_min) + thresh_min
    return gt_prob, gt_thresh, prob_mask, thresh_mask


In [6]:
# ============================================================
# 5. Image preprocessing utilities + helpers shared by both stages
# ============================================================
DET_INPUT_SIZE = 640                              # DBNet input resolution
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def letterbox_image(img: np.ndarray, size: int = DET_INPUT_SIZE):
    """Resize keeping aspect ratio, then pad to (size, size).

    Returns: (padded_image, scale, pad_x, pad_y)
    """
    h, w = img.shape[:2]
    scale = size / max(h, w)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((size, size, 3), dtype=resized.dtype)
    pad_x = (size - new_w) // 2
    pad_y = (size - new_h) // 2
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized
    return canvas, scale, pad_x, pad_y


def transform_polygons(polys: List[np.ndarray], scale: float, pad_x: int, pad_y: int) -> List[np.ndarray]:
    out = []
    for p in polys:
        q = p.copy().astype(np.float32)
        q[:, 0] = q[:, 0] * scale + pad_x
        q[:, 1] = q[:, 1] * scale + pad_y
        out.append(q)
    return out


def to_tensor_norm(img_bgr: np.ndarray) -> torch.Tensor:
    """BGR uint8 -> ImageNet-normalised float CHW tensor (used by DBNet)."""
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = (img - MEAN) / STD
    return torch.from_numpy(img.transpose(2, 0, 1)).contiguous()


def order_quad(pts: np.ndarray) -> np.ndarray:
    """Reorder 4 points as [TL, TR, BR, BL] so they line up with crop_quad's expectations."""
    pts = np.asarray(pts, dtype=np.float32)
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]              # TL
    rect[2] = pts[np.argmax(s)]              # BR
    diff = (pts[:, 1] - pts[:, 0])
    rect[1] = pts[np.argmin(diff)]           # TR
    rect[3] = pts[np.argmax(diff)]           # BL
    return rect


# Recognition crop sizes (kept identical to the recognition-only notebook)
IMG_H = 32
IMG_W_MAX = 512


def crop_quad(image_bgr: np.ndarray, pts: np.ndarray, target_h: int = IMG_H) -> np.ndarray:
    """Perspective-warp a 4-point quadrilateral to a target_h x W strip."""
    pts = pts.astype(np.float32)
    w1 = np.linalg.norm(pts[1] - pts[0]); w2 = np.linalg.norm(pts[2] - pts[3])
    h1 = np.linalg.norm(pts[3] - pts[0]); h2 = np.linalg.norm(pts[2] - pts[1])
    W = max(int((w1 + w2) / 2), 1); H = max(int((h1 + h2) / 2), 1)
    dst = np.array([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(pts, dst)
    crop = cv2.warpPerspective(image_bgr, M, (W, H))
    scale = target_h / max(crop.shape[0], 1)
    new_w = max(int(crop.shape[1] * scale), 1)
    return cv2.resize(crop, (new_w, target_h))


def preprocess_crop(crop_bgr: np.ndarray) -> torch.Tensor:
    """RGB -> [-1, 1] CHW tensor (matches the recognition-only CRNN notebook)."""
    img = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    return torch.from_numpy(img.transpose(2, 0, 1))


In [12]:
# ============================================================
# Recognition dataset — PREDICTED crops from frozen DBNet
# ============================================================
# ============================================================
# Recognition dataset — PREDICTED crops from frozen DBNet
# (placed here because it depends on DBNet, decode_db_polygons,
#  letterbox_image, crop_quad, preprocess_crop, polygon_iou)
# ============================================================
class PredictedCropDataset(Dataset):
    """
    Runs the frozen DBNet once over the full image set at construction time,
    caches all predicted (crop_tensor, label_indices) pairs in RAM, then
    serves them as a standard CTC-ready dataset.

    Label assignment: each predicted quad is matched to the nearest GT quad
    by IoU >= iou_thresh. Unmatched predictions are discarded (no label).
    """
    def __init__(self,
                 img_dir: str,
                 label_dir: str,
                 detector: "DBNet",
                 device: torch.device,
                 iou_thresh: float = 0.5,
                 max_label_len: int = 25,
                 augment: bool = False):
        self.augment = augment
        self.samples = []

        image_paths = sorted(
            glob.glob(os.path.join(img_dir, "*.jpg")) +
            glob.glob(os.path.join(img_dir, "*.png"))
        )

        detector.eval()
        matched = skipped_label = skipped_iou = 0

        print(f"  [pred-crop] building {os.path.basename(img_dir)} "
              f"({len(image_paths)} images) ...")

        with torch.no_grad():
            for ip in tqdm(image_paths, desc=f"  det {os.path.basename(img_dir)}",
                           leave=False):
                img = cv2.imread(ip)
                if img is None:
                    continue
                h0, w0 = img.shape[:2]

                canvas, scale, px, py = letterbox_image(img, DET_INPUT_SIZE)
                tensor = to_tensor_norm(canvas).unsqueeze(0).to(device)
                prob, _, _ = detector(tensor)
                decoded = decode_db_polygons(prob)[0]

                pred_quads = []
                for quad, _score in decoded:
                    q = quad.copy()
                    q[:, 0] = (q[:, 0] - px) / max(scale, 1e-6)
                    q[:, 1] = (q[:, 1] - py) / max(scale, 1e-6)
                    q[:, 0] = np.clip(q[:, 0], 0, w0 - 1)
                    q[:, 1] = np.clip(q[:, 1], 0, h0 - 1)
                    pred_quads.append(q.astype(np.float32))

                if not pred_quads:
                    continue

                idx = _img_idx_from_name(ip)
                gt_recs = parse_annotation(os.path.join(label_dir, f"gt_{idx}.txt"))
                if not gt_recs:
                    continue
                gt_polys = [r[0] for r in gt_recs]
                gt_texts = [r[1] for r in gt_recs]
                gt_used  = [False] * len(gt_polys)

                for pred_q in pred_quads:
                    best_iou = 0.0
                    best_j   = -1
                    for j, gp in enumerate(gt_polys):
                        if gt_used[j]:
                            continue
                        iou = polygon_iou(pred_q, gp)
                        if iou > best_iou:
                            best_iou, best_j = iou, j

                    if best_j < 0 or best_iou < iou_thresh:
                        skipped_iou += 1
                        continue

                    transcript = gt_texts[best_j]
                    if transcript in ("###", "***", ""):
                        skipped_label += 1
                        continue

                    label_idx = text_to_indices(transcript)
                    if not label_idx or len(label_idx) > max_label_len:
                        skipped_label += 1
                        continue

                    gt_used[best_j] = True

                    try:
                        crop = crop_quad(img, pred_q)
                    except Exception:
                        skipped_label += 1
                        continue

                    t = preprocess_crop(crop)
                    if t.shape[2] > IMG_W_MAX:
                        t = t[:, :, :IMG_W_MAX]

                    self.samples.append((t, label_idx))
                    matched += 1

        print(f"  [pred-crop] {os.path.basename(img_dir)}: "
              f"{matched} matched crops  "
              f"(dropped {skipped_iou} no-IoU-match, {skipped_label} bad-label)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        t, label = self.samples[i]
        if self.augment:
            img_np = ((t.permute(1, 2, 0).numpy() * 0.5 + 0.5) * 255).clip(0, 255).astype(np.uint8)
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
            a = random.uniform(0.7, 1.3); b = random.randint(-20, 20)
            img_np = np.clip(img_np.astype(np.float32) * a + b, 0, 255).astype(np.uint8)
            if random.random() < 0.3:
                k = random.choice([3, 5])
                img_np = cv2.GaussianBlur(img_np, (k, k), 0)
            t = preprocess_crop(img_np)
            if t.shape[2] > IMG_W_MAX:
                t = t[:, :, :IMG_W_MAX]
        return t, label


# rec_collate defined here — PredictedCropDataset follows after model definitions below
def rec_collate(batch):
    """Variable-width padded batch for CTC training."""
    batch = sorted(batch, key=lambda x: -x[0].shape[2])
    imgs, labels = zip(*batch)
    max_w = imgs[0].shape[2]
    padded = [F.pad(img, (0, max_w - img.shape[2]), value=-1.0) for img in imgs]
    images         = torch.stack(padded)
    targets        = torch.tensor([i for s in labels for i in s], dtype=torch.long)
    target_lengths = torch.tensor([len(s) for s in labels], dtype=torch.long)
    input_lengths  = torch.tensor([max(1, img.shape[2] // 4) for img in imgs], dtype=torch.long)
    return images, targets, input_lengths, target_lengths

In [13]:
# ============================================================
# 7. DBNet model: ResNet-18 + FPN + dual heads (prob & threshold)
#    Differentiable binarization (Liao et al., AAAI'20).
# ============================================================
class FPN(nn.Module):
    """Lightweight FPN matched to ResNet-18 stage channels (64,128,256,512)."""
    def __init__(self, in_channels=(64, 128, 256, 512), inner_channels: int = 256):
        super().__init__()
        c2, c3, c4, c5 = in_channels
        self.lat5 = nn.Conv2d(c5, inner_channels, 1)
        self.lat4 = nn.Conv2d(c4, inner_channels, 1)
        self.lat3 = nn.Conv2d(c3, inner_channels, 1)
        self.lat2 = nn.Conv2d(c2, inner_channels, 1)
        self.smooth5 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)
        self.smooth4 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)
        self.smooth3 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)
        self.smooth2 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)

    def forward(self, c2, c3, c4, c5):
        p5 = self.lat5(c5)
        p4 = self.lat4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode="bilinear", align_corners=False)
        p3 = self.lat3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="bilinear", align_corners=False)
        p2 = self.lat2(c2) + F.interpolate(p3, size=c2.shape[-2:], mode="bilinear", align_corners=False)
        p5 = self.smooth5(p5); p4 = self.smooth4(p4)
        p3 = self.smooth3(p3); p2 = self.smooth2(p2)
        size = p2.shape[-2:]
        p3 = F.interpolate(p3, size=size, mode="bilinear", align_corners=False)
        p4 = F.interpolate(p4, size=size, mode="bilinear", align_corners=False)
        p5 = F.interpolate(p5, size=size, mode="bilinear", align_corners=False)
        return torch.cat([p2, p3, p4, p5], dim=1)        # (B, inner, H/4, W/4)


class DBNet(nn.Module):
    def __init__(self, k: int = 50, pretrained: bool = True):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        backbone = resnet18(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1   # /4
        self.layer2 = backbone.layer2   # /8
        self.layer3 = backbone.layer3   # /16
        self.layer4 = backbone.layer4   # /32
        self.fpn = FPN()
        inner = 256

        def _head():
            return nn.Sequential(
                nn.Conv2d(inner, inner // 4, 3, padding=1, bias=False),
                nn.BatchNorm2d(inner // 4), nn.ReLU(True),
                nn.ConvTranspose2d(inner // 4, inner // 4, 2, stride=2),
                nn.BatchNorm2d(inner // 4), nn.ReLU(True),
                nn.ConvTranspose2d(inner // 4, 1, 2, stride=2),
            )

        self.prob_head   = _head()
        self.thresh_head = _head()
        self.k = k

    def forward(self, x):
        c1 = self.stem(x)
        c2 = self.layer1(c1)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        feat   = self.fpn(c2, c3, c4, c5)
        prob   = torch.sigmoid(self.prob_head(feat))         # (B, 1, H, W)
        thresh = torch.sigmoid(self.thresh_head(feat))       # (B, 1, H, W)
        binary = torch.sigmoid(self.k * (prob - thresh))     # differentiable binarization
        return prob, thresh, binary


In [14]:
# ============================================================
# 9. DBNet post-processing: prob map -> 4-point quad polygons
# ============================================================
@torch.no_grad()
def decode_db_polygons(prob_map: torch.Tensor,
                       prob_thresh: float = 0.3,
                       box_thresh: float = 0.55,
                       min_size: float = 3.0,
                       max_candidates: int = 1000,
                       unclip_ratio: float = 1.5):
    """Convert DBNet probability maps to lists of (quad, score) per image.

    Coordinates are returned in the detector's input space (det_size x det_size).
    """
    results = []
    prob_np = prob_map.detach().squeeze(1).cpu().numpy()      # (B, H, W)
    binary  = (prob_np >= prob_thresh).astype(np.uint8)
    for prob, b in zip(prob_np, binary):
        contours, _ = cv2.findContours(b, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        polys = []
        for cnt in contours[:max_candidates]:
            if cnt.shape[0] < 4:
                continue
            epsilon = 0.005 * cv2.arcLength(cnt, True)
            approx  = cv2.approxPolyDP(cnt, epsilon, True).reshape(-1, 2)
            if approx.shape[0] < 4:
                rect = cv2.minAreaRect(cnt)
                approx = cv2.boxPoints(rect).astype(np.float32)
            mask = np.zeros_like(prob, dtype=np.uint8)
            cv2.fillPoly(mask, [approx.astype(np.int32)], 1)
            if mask.sum() == 0:
                continue
            score = float(prob[mask == 1].mean())
            if score < box_thresh:
                continue
            try:
                p = Polygon(approx.astype(np.float64))
                if (not p.is_valid) or p.area < min_size:
                    continue
                distance = p.area * unclip_ratio / max(p.length, 1e-6)
            except Exception:
                continue
            pco = pyclipper.PyclipperOffset()
            try:
                pco.AddPath(np.round(approx).astype(int).tolist(),
                            pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
                expanded = pco.Execute(distance)
            except Exception:
                continue
            if not expanded:
                continue
            poly = np.asarray(expanded[0], dtype=np.float32)
            if poly.shape[0] < 4:
                continue
            # Reduce to a 4-point quadrilateral so it lines up with crop_quad's expectation.
            rect = cv2.minAreaRect(poly)
            quad = cv2.boxPoints(rect).astype(np.float32)
            quad = order_quad(quad)
            polys.append((quad, score))
        results.append(polys)
    return results


In [15]:
# ============================================================
# 10. CRNN recogniser  (identical to the recognition-only notebook)
# ============================================================
class CRNN(nn.Module):
    """CNN + BiLSTM + CTC (Shi et al., 2015) -- kept verbatim from the original notebook."""
    def __init__(self, num_classes: int, hidden_size: int = 256, n_rnn_layers: int = 2):
        super().__init__()

        def _block(ci, co, pool=None):
            layers = [nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co),
                      nn.LeakyReLU(0.2, inplace=True)]
            if pool == "full":
                layers.append(nn.MaxPool2d(2, 2))
            elif pool == "height":
                layers.append(nn.MaxPool2d((2, 1), (2, 1)))
            return nn.Sequential(*layers)

        self.cnn = nn.Sequential(
            _block(3,   64,  "full"),
            _block(64,  128, "full"),
            _block(128, 256),
            _block(256, 256, "height"),
            _block(256, 512),
            _block(512, 512, "height"),
            _block(512, 512, "height"),
            nn.Conv2d(512, 512, (1, 1)),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.rnn = nn.LSTM(512, hidden_size, num_layers=n_rnn_layers,
                           batch_first=False, bidirectional=True,
                           dropout=0.2 if n_rnn_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        feat = self.cnn(x)
        b, c, h, w = feat.size()
        if h > 1:
            feat = F.adaptive_avg_pool2d(feat, (1, w))
        feat = feat.squeeze(2).permute(2, 0, 1)              # (W, B, C)
        out, _ = self.rnn(feat)
        return F.log_softmax(self.fc(out), dim=2)            # (T, B, C)


def ctc_greedy_decode(log_probs: torch.Tensor) -> List[str]:
    _, idx = log_probs.max(2)
    out = []
    for seq in idx.permute(1, 0):
        seq = seq.tolist(); prev = None; chars = []
        for i in seq:
            if i != prev and i != 0:
                chars.append(i)
            prev = i
        out.append(indices_to_text(chars))
    return out


In [16]:
# ============================================================
# 11. End-to-end inference pipeline:
#      raw image  ->  DBNet  ->  predicted polygons  ->  CRNN  ->  strings
# ============================================================
class SpotterPipeline:
    def __init__(self, detector: DBNet, recognizer: CRNN, device: torch.device,
                 det_size: int = DET_INPUT_SIZE,
                 prob_thresh: float = 0.3,
                 box_thresh: float = 0.55,
                 unclip_ratio: float = 1.5,
                 min_quad_size: int = 4):
        self.detector   = detector.to(device)
        self.recognizer = recognizer.to(device)
        self.device       = device
        self.det_size     = det_size
        self.prob_thresh  = prob_thresh
        self.box_thresh   = box_thresh
        self.unclip_ratio = unclip_ratio
        self.min_quad_size = min_quad_size

    def detect(self, image_bgr: np.ndarray):
        """Run only the detector; return predicted quads in *original* image coords."""
        h0, w0 = image_bgr.shape[:2]
        canvas, scale, px, py = letterbox_image(image_bgr, self.det_size)
        tensor = to_tensor_norm(canvas).unsqueeze(0).to(self.device)
        prob, _, _ = self.detector(tensor)
        polys_per_image = decode_db_polygons(
            prob, self.prob_thresh, self.box_thresh,
            unclip_ratio=self.unclip_ratio,
        )
        out = []
        for quad, score in polys_per_image[0]:
            quad_orig = quad.copy()
            quad_orig[:, 0] = (quad_orig[:, 0] - px) / max(scale, 1e-6)
            quad_orig[:, 1] = (quad_orig[:, 1] - py) / max(scale, 1e-6)
            quad_orig[:, 0] = np.clip(quad_orig[:, 0], 0, w0 - 1)
            quad_orig[:, 1] = np.clip(quad_orig[:, 1], 0, h0 - 1)
            out.append((quad_orig.astype(np.float32), score))
        return out

    @torch.no_grad()
    def __call__(self, image_bgr: np.ndarray) -> List[Tuple[np.ndarray, str, float]]:
        self.detector.eval(); self.recognizer.eval()
        det_results = self.detect(image_bgr)
        if not det_results:
            return []
        spotted = []
        for quad, score in det_results:
            try:
                crop = crop_quad(image_bgr, quad.astype(np.float32))
            except Exception:
                continue
            if crop.shape[1] < self.min_quad_size:
                continue
            t = preprocess_crop(crop).unsqueeze(0).to(self.device)
            if t.shape[3] > IMG_W_MAX:
                t = t[:, :, :, :IMG_W_MAX]
            log_probs = self.recognizer(t)
            text = ctc_greedy_decode(log_probs)[0]
            spotted.append((quad, text, score))
        return spotted


In [17]:
# ============================================================
# End-to-end evaluation: Det + E2E P/R/H-mean, CER/WER (IoU-matched)
# Same protocol as vintext-e2e-paddle-ppocr(12epochs).ipynb
# ============================================================
def polygon_iou(p1: np.ndarray, p2: np.ndarray) -> float:
    try:
        a = Polygon(p1); b = Polygon(p2)
        if not a.is_valid: a = a.buffer(0)
        if not b.is_valid: b = b.buffer(0)
        if a.is_empty or b.is_empty:
            return 0.0
        inter = a.intersection(b).area
        union = a.union(b).area
        return float(inter / union) if union > 0 else 0.0
    except Exception:
        return 0.0


def _prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    h = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, h


def evaluate_endtoend(pipeline: SpotterPipeline,
                      img_dir: str, label_dir: str,
                      iou_thresh: float = 0.5,
                      max_images: Optional[int] = None) -> dict:
    pipeline.detector.eval()
    pipeline.recognizer.eval()
    image_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")) +
                         glob.glob(os.path.join(img_dir, "*.png")))
    if max_images:
        image_paths = image_paths[:max_images]

    TP = FP = FN = 0
    det_TP = det_FP = det_FN = 0
    char_edits = char_total = 0
    word_edits = word_total = 0
    rec_matched = 0
    n_images = 0

    for ip in tqdm(image_paths, desc=f"E2E eval [{os.path.basename(img_dir)}]"):
        idx = _img_idx_from_name(ip)
        gt = parse_annotation(os.path.join(label_dir, f"gt_{idx}.txt"))
        if not gt:
            continue
        gt_polys = [g[0] for g in gt]
        gt_texts = [normalize_for_match(g[1]) for g in gt]
        gt_dontcare = [t in ("###", "***", "") for t in gt_texts]

        img = cv2.imread(ip)
        if img is None:
            fn_count = sum(1 for d in gt_dontcare if not d)
            FN += fn_count; det_FN += fn_count
            continue

        preds = pipeline(img)
        n_images += 1
        gt_used = [False] * len(gt_polys)

        for poly, text, score in preds:
            best_iou = 0.0; best_j = -1
            for j, gp in enumerate(gt_polys):
                if gt_used[j]:
                    continue
                iou = polygon_iou(poly, gp)
                if iou > best_iou:
                    best_iou, best_j = iou, j

            iou_matched = (best_j >= 0 and best_iou >= iou_thresh)
            if iou_matched and gt_dontcare[best_j]:
                gt_used[best_j] = True
                continue

            if iou_matched:
                gt_used[best_j] = True
                det_TP += 1
                pred_t = normalize_for_match(text)
                gt_t = gt_texts[best_j]
                char_edits += editdistance.eval(pred_t, gt_t)
                char_total += max(len(gt_t), 1)
                gt_w = gt_t.split() if gt_t.split() else [gt_t]
                pr_w = pred_t.split() if pred_t.split() else [pred_t]
                word_edits += editdistance.eval(pr_w, gt_w)
                word_total += max(len(gt_w), 1)
                rec_matched += 1
                if pred_t == gt_t:
                    TP += 1
                else:
                    FP += 1
            else:
                det_FP += 1; FP += 1

        for j, used in enumerate(gt_used):
            if used or gt_dontcare[j]:
                continue
            det_FN += 1; FN += 1

    P, R, H = _prf(TP, FP, FN)
    Pd, Rd, Hd = _prf(det_TP, det_FP, det_FN)
    cer = char_edits / char_total if char_total else 0.0
    wer = word_edits / word_total if word_total else 0.0
    return dict(
        precision=P, recall=R, hmean=H,
        det_precision=Pd, det_recall=Rd, det_hmean=Hd,
        cer=cer, wer=wer, rec_matched=rec_matched,
        TP=TP, FP=FP, FN=FN,
        det_TP=det_TP, det_FP=det_FP, det_FN=det_FN,
        images=n_images,
    )


def print_e2e_results(tag: str, res: dict):
    print(f"  [{tag}] Detection (IoU>=0.5): P={res['det_precision']:.4f}  R={res['det_recall']:.4f}  H-mean={res['det_hmean']:.4f}")
    print(f"  [{tag}] End-to-end:           P={res['precision']:.4f}  R={res['recall']:.4f}  H-mean={res['hmean']:.4f}")
    print(f"  [{tag}] Recognition (IoU-matched): CER={res['cer']:.4f}  WER={res['wer']:.4f}  (n={res['rec_matched']})")


In [18]:
# ============================================================
# Load frozen pretrained DBNet (.pth from detection notebook)
# ============================================================
DET_WORK_DIR = "/kaggle/working/dbnet_det_vintext"
DET_PTH_CANDIDATES = [
    os.path.join(DET_WORK_DIR, "dbnet_resnet18_det_vintext_12epochs.pth"),
    os.path.join(DET_WORK_DIR, "dbnet_det_best.pth"),
    "/kaggle/input/models/khnhtraan/detect/pytorch/default/1/dbnet_resnet18_det_vintext_12epochs.pth",
    "/kaggle/input/models/khnhtraan/detect1/pytorch/default/1/dbnet_resnet18_det_vintext_12epochs.pth",
]


def _find_det_pth():
    for p in DET_PTH_CANDIDATES:
        if os.path.isfile(p):
            return p
    for base in glob.glob("/kaggle/input/*"):
        for root, _, files in os.walk(base):
            for name in ("dbnet_resnet18_det_vintext_12epochs.pth", "dbnet_det_best.pth"):
                if name in files:
                    return os.path.join(root, name)
    return None


DET_PTH = _find_det_pth()
if DET_PTH is None:
    raise FileNotFoundError(
        "Pretrained DBNet .pth not found. Run dbnet-det-vintext-12epochs.ipynb first "
        "and add the output as a Kaggle dataset."
    )
print(f"Loading frozen detector: {DET_PTH}")

detector = DBNet(pretrained=False).to(DEVICE)
try:
    ckpt = torch.load(DET_PTH, map_location=DEVICE, weights_only=False)
except TypeError:
    ckpt = torch.load(DET_PTH, map_location=DEVICE)

state = ckpt.get("detector_state", ckpt)
detector.load_state_dict(state, strict=True)
for p in detector.parameters():
    p.requires_grad = False
detector.eval()
print("Detector frozen (eval, no grad).")


Loading frozen detector: /kaggle/input/models/khnhtraan/detect1/pytorch/default/1/dbnet_resnet18_det_vintext_12epochs.pth
Detector frozen (eval, no grad).


In [19]:
# ============================================================
# Crop-level recognition accuracy (exact NFD string match on GT crops)
# ============================================================
@torch.no_grad()
def recognition_crop_accuracy(recognizer: CRNN, loader,
                              max_batches: Optional[int] = None) -> float:
    recognizer.eval()
    correct = total = 0
    for bi, batch in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        crops, targets, in_lens, tgt_lens = [x.to(DEVICE) for x in batch]
        log_probs = recognizer(crops)
        pred_texts = ctc_greedy_decode(log_probs)
        offset = 0
        for i, tl in enumerate(tgt_lens.cpu().tolist()):
            gt_text = indices_to_text(targets[offset:offset + tl].cpu().tolist())
            offset += tl
            if normalize_for_match(pred_texts[i]) == normalize_for_match(gt_text):
                correct += 1
            total += 1
    return correct / max(total, 1)


In [20]:
# ============================================================
# Recognition-only training (CRNN on GT crops; frozen DBNet at inference)
# ============================================================
ACC_MAX_BATCHES_TRAIN = 50   # subset of train loader for per-epoch crop accuracy


def train_recognition(recognizer: CRNN,
                      train_loader, val_loader,
                      detector: DBNet,
                      num_epochs: int = 12,
                      lr: float = 1e-4,
                      grad_clip: float = 5.0,
                      e2e_eval_fn=None,
                      ckpt_path: Optional[str] = None):
    ctc_loss_fn = nn.CTCLoss(blank=0, zero_infinity=True).to(DEVICE)
    optimizer = AdamW(recognizer.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    history = dict(
        train_loss=[], val_loss=[], train_acc=[], val_acc=[],
        val_e2e_hmean=[], val_det_hmean=[],
    )
    if ckpt_path is None:
        ckpt_path = os.path.join(OUT_DIR, "dbnet_crnn_e2e_best.pth")

    best_metric = -1.0
    for epoch in range(1, num_epochs + 1):
        recognizer.train()
        detector.eval()
        running = 0.0
        pbar = tqdm(train_loader, desc=f"E{epoch}/{num_epochs} rec train")
        for batch in pbar:
            crops, targets, in_lens, tgt_lens = [x.to(DEVICE, non_blocking=True) for x in batch]
            log_probs = recognizer(crops)
            T_steps = log_probs.size(0)
            in_lens = torch.clamp(in_lens, max=T_steps)
            loss = ctc_loss_fn(log_probs, targets, in_lens, tgt_lens)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(recognizer.parameters(), grad_clip)
            optimizer.step()
            running += loss.item()
            pbar.set_postfix(ctc=f"{loss.item():.3f}")
        scheduler.step()
        history["train_loss"].append(running / max(len(train_loader), 1))

        recognizer.eval()
        v_loss = 0.0; steps = 0
        with torch.no_grad():
            for batch in val_loader:
                crops, targets, in_lens, tgt_lens = [x.to(DEVICE) for x in batch]
                log_probs = recognizer(crops)
                T_steps = log_probs.size(0)
                in_lens = torch.clamp(in_lens, max=T_steps)
                v_loss += ctc_loss_fn(log_probs, targets, in_lens, tgt_lens).item()
                steps += 1
        v_loss /= max(steps, 1)
        history["val_loss"].append(v_loss)

        train_acc = recognition_crop_accuracy(
            recognizer, train_loader, max_batches=ACC_MAX_BATCHES_TRAIN,
        )
        val_acc = recognition_crop_accuracy(recognizer, val_loader)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        e2e_h = None
        res = None
        if e2e_eval_fn is not None:
            res = e2e_eval_fn()
            e2e_h = res["hmean"]
            history["val_e2e_hmean"].append(e2e_h)
            history["val_det_hmean"].append(res["det_hmean"])

        track = e2e_h if e2e_h is not None else val_acc
        msg = (f"Epoch {epoch}: train_ctc={history['train_loss'][-1]:.4f}  val_ctc={v_loss:.4f}  "
               f"| train_acc={train_acc:.4f}  val_acc={val_acc:.4f}")
        if e2e_h is not None:
            msg += f"  | val_e2e_H={e2e_h:.4f}  CER={res['cer']:.4f}"
        print(msg)

        if track > best_metric:
            best_metric = track
            torch.save({
                "epoch": epoch,
                "seed": SEED,
                "framework": "pytorch_dbnet_crnn_e2e_vintext",
                "det_pth_source": DET_PTH,
                "detector_state": detector.state_dict(),
                "recognizer_state": recognizer.state_dict(),
                "history": history,
                "best_metric": best_metric,
            }, ckpt_path)
            print(f"    saved best -> {ckpt_path}")

    return history, ckpt_path


In [24]:
# ============================================================
# Build predicted-crop datasets, train CRNN ( epoch)
# ============================================================
print("=== Building predicted-crop recognition datasets ===")
rec_train = PredictedCropDataset(TRAIN_DIR, LABEL_DIR, detector, DEVICE, augment=True)
rec_val   = PredictedCropDataset(VAL_DIR,   LABEL_DIR, detector, DEVICE, augment=False)

rec_train_loader = DataLoader(rec_train, batch_size=64, shuffle=True,
                              collate_fn=rec_collate, num_workers=2,
                              pin_memory=True, drop_last=True)
rec_val_loader   = DataLoader(rec_val, batch_size=64, shuffle=False,
                              collate_fn=rec_collate, num_workers=2,
                              pin_memory=True)

print(f"Rec batches: train={len(rec_train_loader)}  val={len(rec_val_loader)}")

recognizer = CRNN(NUM_CLASSES).to(DEVICE)
print(f"Recognizer params: {sum(p.numel() for p in recognizer.parameters()):,}")

PRETRAINED_REC = "/kaggle/input/crnn-best/crnn_best.pth"
if os.path.exists(PRETRAINED_REC):
    try:
        ck = torch.load(PRETRAINED_REC, map_location=DEVICE, weights_only=False)
    except TypeError:
        ck = torch.load(PRETRAINED_REC, map_location=DEVICE)
    state = ck.get("model_state", ck.get("recognizer_state", ck))
    # Sanity-check fc layer matches current alphabet before loading
    if "fc.weight" in state and state["fc.weight"].shape[0] != NUM_CLASSES:
        print(f"  WARNING: pretrained fc size {state['fc.weight'].shape[0]} != "
              f"NUM_CLASSES {NUM_CLASSES}. Skipping warm-start.")
    else:
        recognizer.load_state_dict(state, strict=False)
        print(f"Warm-started recognizer from {PRETRAINED_REC}")

NUM_EPOCHS = 50
CKPT_BEST  = os.path.join(OUT_DIR, "dbnet_crnn_e2e_best.pth")

def _val_e2e_eval():
    pipe = SpotterPipeline(detector, recognizer, DEVICE)
    return evaluate_endtoend(pipe, VAL_DIR, LABEL_DIR, max_images=80)

print("\n=== Training CRNN on predicted crops (frozen DBNet) ===")
history, best_path = train_recognition(
    recognizer, rec_train_loader, rec_val_loader, detector,
    num_epochs=NUM_EPOCHS, e2e_eval_fn=_val_e2e_eval,
    ckpt_path=CKPT_BEST,
)

REC_PTH = os.path.join(OUT_DIR, "dbnet_crnn_e2e_vintext_50epochs.pth")
torch.save(torch.load(best_path, map_location="cpu", weights_only=False), REC_PTH)
print(f"\nExported E2E bundle -> {REC_PTH}")

=== Building predicted-crop recognition datasets ===
  [pred-crop] building train_images (1200 images) ...


  [pred-crop] train_images: 13897 matched crops  (dropped 1979 no-IoU-match, 77 bad-label)
  [pred-crop] building val_image (300 images) ...


  [pred-crop] val_image: 3485 matched crops  (dropped 717 no-IoU-match, 23 bad-label)
Rec batches: train=217  val=55
Recognizer params: 10,337,130

=== Training CRNN on predicted crops (frozen DBNet, 12 epochs) ===


E2E eval [val_image]: 100%|██████████| 80/80 [00:12<00:00,  6.39it/s]


Epoch 1: train_ctc=4.5462  val_ctc=3.8030  | train_acc=0.0000  val_acc=0.0000  | val_e2e_H=0.0000  CER=0.9610
    saved best -> /kaggle/working/dbnet_crnn_e2e_vintext/dbnet_crnn_e2e_best.pth


...

Epoch 48: train_ctc=0.0320  val_ctc=0.9730  | train_acc=0.1384  val_acc=0.1225  | val_e2e_H=0.3101  CER=0.2341


E2E eval [val_image]: 100%|██████████| 80/80 [00:12<00:00,  6.19it/s]


Epoch 49: train_ctc=0.0310  val_ctc=0.9711  | train_acc=0.1094  val_acc=0.1105  | val_e2e_H=0.3163  CER=0.2313


E2E eval [val_image]: 100%|██████████| 80/80 [00:12<00:00,  6.32it/s]


Epoch 50: train_ctc=0.0309  val_ctc=0.9780  | train_acc=0.1750  val_acc=0.1242  | val_e2e_H=0.3149  CER=0.2315

Exported E2E bundle -> /kaggle/working/dbnet_crnn_e2e_vintext/dbnet_crnn_e2e_vintext_12epochs.pth


In [25]:
# ============================================================
# Training curves — CTC loss & crop recognition accuracy
# ============================================================
_hist = history if "history" in dir() and history.get("train_loss") else None
if _hist is None:
    _hist = torch.load(best_path, map_location="cpu", weights_only=False).get("history", {})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(_hist["train_loss"]) + 1)

axes[0].plot(ep, _hist["train_loss"], "o-", label="train", linewidth=2, markersize=5)
axes[0].plot(ep, _hist["val_loss"], "o-", label="val", linewidth=2, markersize=5)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("CTC loss")
axes[0].set_title("Train / Val loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, _hist["train_acc"], "o-", label="train", linewidth=2, markersize=5)
axes[1].plot(ep, _hist["val_acc"], "o-", label="val", linewidth=2, markersize=5)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("E2E Recog Train / Val accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curve_path = os.path.join(OUT_DIR, "training_curves.png")
plt.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Training curves saved -> {curve_path}")


Training curves saved -> /kaggle/working/dbnet_crnn_e2e_vintext/training_curves.png


In [26]:
# ============================================================
# Final evaluation: val + test (Det P/R/H, E2E P/R/H, CER/WER)
# ============================================================
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
recognizer.load_state_dict(ckpt["recognizer_state"])
pipeline = SpotterPipeline(detector, recognizer, DEVICE)

final_results = {}
for tag, img_dir in [("Validation", VAL_DIR), ("Test", TEST_DIR)]:
    print(f"\n--- {tag} ---")
    res = evaluate_endtoend(pipeline, img_dir, LABEL_DIR, iou_thresh=0.5)
    final_results[tag] = res
    print_e2e_results(tag, res)
    with open(os.path.join(OUT_DIR, f"e2e_metrics_{tag.lower()}.json"), "w") as f:
        json.dump(res, f, indent=2)

print("\n" + "=" * 88)
print(f"  {'Split':<12} {'Det H':>7} {'E2E H':>7} {'CER':>7} {'WER':>7}")
print("  " + "-" * 44)
for tag, r in final_results.items():
    print(f"  {tag:<12} {r['det_hmean']:>7.4f} {r['hmean']:>7.4f} {r['cer']:>7.4f} {r['wer']:>7.4f}")
print("=" * 88)



--- Validation ---


E2E eval [val_image]: 100%|██████████| 300/300 [00:44<00:00,  6.68it/s]


  [Validation] Detection (IoU>=0.5): P=0.8294  R=0.4842  H-mean=0.6115
  [Validation] End-to-end:           P=0.3444  R=0.2805  H-mean=0.3092
  [Validation] Recognition (IoU-matched): CER=0.2265  WER=0.5848  (n=3487)

--- Test ---


E2E eval [test_image]: 100%|██████████| 500/500 [01:08<00:00,  7.33it/s]

  [Test] Detection (IoU>=0.5): P=0.8535  R=0.5301  H-mean=0.6540
  [Test] End-to-end:           P=0.3405  R=0.3104  H-mean=0.3247
  [Test] Recognition (IoU-matched): CER=0.2430  WER=0.6015  (n=5337)

  Split          Det H   E2E H     CER     WER
  --------------------------------------------
  Validation    0.6115  0.3092  0.2265  0.5848
  Test          0.6540  0.3247  0.2430  0.6015


In [ ]:
# ============================================================
# Sample E2E inference — GT (green) vs pred boxes + text (cyan)
# ============================================================
from PIL import Image, ImageDraw, ImageFont
import subprocess

ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
recognizer.load_state_dict(ckpt["recognizer_state"])
pipeline = SpotterPipeline(detector, recognizer, DEVICE)

FONT_PATH = "/kaggle/working/NotoSans-Regular.ttf"
if not os.path.exists(FONT_PATH):
    try:
        subprocess.run([
            "wget", "-q", "-O", FONT_PATH,
            "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Regular.ttf",
        ], check=True)
    except Exception as e:
        print(f"(font download skipped: {e})")


def _font(size: int):
    try:
        return ImageFont.truetype(FONT_PATH, size)
    except Exception:
        return ImageFont.load_default()


def draw_text_pil(canvas_bgr, text, position, color_rgb, font_size=18, bg_rgb=(0, 0, 0)):
    pil = Image.fromarray(cv2.cvtColor(canvas_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil)
    font = _font(font_size)
    bbox = draw.textbbox(position, text, font=font)
    pad = 2
    draw.rectangle([bbox[0] - pad, bbox[1] - pad, bbox[2] + pad, bbox[3] + pad], fill=bg_rgb)
    draw.text(position, text, font=font, fill=color_rgb)
    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)


def visualize_e2e(pipe: SpotterPipeline, img_dir: str, label_dir: str,
                    n_samples: int = 6, out_subdir: str = "e2e_inference_samples"):
    out_viz = os.path.join(OUT_DIR, out_subdir)
    os.makedirs(out_viz, exist_ok=True)
    image_paths = sorted(
        glob.glob(os.path.join(img_dir, "*.jpg")) +
        glob.glob(os.path.join(img_dir, "*.png"))
    )[:n_samples]

    for ip in image_paths:
        img = cv2.imread(ip)
        if img is None:
            continue
        idx = _img_idx_from_name(ip)
        gt = parse_annotation(os.path.join(label_dir, f"gt_{idx}.txt"))
        preds = pipe(img)

        canvas = img.copy()
        font_size = max(14, img.shape[0] // 55)
        for poly, txt in gt:
            cv2.polylines(canvas, [poly.astype(np.int32)], True, (0, 200, 0), 2)
            x = int(poly[:, 0].min()); y = int(poly[:, 1].min())
            canvas = draw_text_pil(
                canvas, f"GT: {txt}",
                (x, max(font_size + 4, y - 2 * font_size - 6)),
                color_rgb=(0, 230, 0), font_size=font_size,
            )
        for poly, text, score in preds:
            cv2.polylines(canvas, [poly.astype(np.int32)], True, (255, 200, 0), 2)
            x = int(poly[:, 0].min()); y = int(poly[:, 1].min())
            canvas = draw_text_pil(
                canvas, f"PR: {text} ({score:.2f})",
                (x, max(font_size + 4, y - font_size - 2)),
                color_rgb=(0, 215, 255), font_size=font_size, bg_rgb=(20, 20, 80),
            )

        fig, axes = plt.subplots(1, 2, figsize=(22, 11))
        axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Original — {os.path.basename(ip)}")
        axes[0].axis("off")
        axes[1].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
        axes[1].set_title("GT (green) vs Predicted (cyan)")
        axes[1].axis("off")
        plt.tight_layout()
        save_path = os.path.join(out_viz, f"e2e_{Path(ip).stem}.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  saved -> {save_path}")
    print(f"\nE2E sample visualizations -> {out_viz}")


visualize_e2e(pipeline, TEST_DIR, LABEL_DIR, n_samples=6)
